# 📦 Demand Forecasting & Inventory Optimization Engine
## Milestone 2: Forecast Model Development

**Notebook Outline:**
1. Setup & Data Loading
2. Product Selection & Time Series Visualization
3. Feature Engineering
4. Train / Validation / Test Split (Time-Based)
5. Baseline Model — Naive Forecast
6. Baseline Model — Moving Average
7. Evaluation & Comparison
8. Data Preparation for Global Model (Log-Transform)
9. Advanced Feature Engineering
10. Train / Val / Test Split + Target Encoding
11. Model Training (XGBoost, Random Forest, Ridge, Ensemble)
12. Final Evaluation on Test Set
13. Model Serialization & Export
14. Export Forecasts for Inventory Optimization

---
## 1. Setup & Data Loading <a id='1'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1.1  Import Libraries
# ══════════════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
np.random.seed(42)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False
})
COLORS = sns.color_palette('muted', 10)

print('✅ Libraries imported')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1.2  Load Dataset
# ══════════════════════════════════════════════════════════════════════════════
df = pd.read_csv('weekly.csv', encoding='utf-8-sig')
df.columns = [
    'Year', 'Week', 'ItemID', 'Qty', 'ItemName', 'BrandID', 'BrandName',
    'MasterBrandID', 'MasterBrandName', 'UOM', 'Factor',
    'Avg_Daily_Demand', 'Safety_Stock', 'ROP', 'Avg_UnitPrice',
    'Total_Promo', 'Total_CashDiscount', 'Total_ManualDiscount',
    'Total_Taxes', 'Total_Revenue'
]
df['Date'] = pd.to_datetime(
    df['Year'].astype(str) + '-W' + df['Week'].astype(str).str.zfill(2) + '-1',
    format='%G-W%V-%u'
)

print(f'✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Date range : {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'   Products   : {df["ItemID"].nunique()}')
print(f'   Weeks      : {df["Week"].nunique()}')

---
## 2. Product Selection & Time Series Visualization <a id='2'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2.1  Identify Top-Selling Products
# ══════════════════════════════════════════════════════════════════════════════
product_sales = df.groupby(['ItemID', 'ItemName'])['Qty'].sum() \
    .sort_values(ascending=False).reset_index().head(10)
product_sales.columns = ['ItemID', 'ItemName', 'Total_Qty']

print('🏆 Top 10 Products by Total Sales:')
print('-' * 60)
for i, row in product_sales.iterrows():
    marker = ' 👈 SELECTED' if i == 0 else ''
    print(f"  {i+1:>2}. {row['ItemName']:<35} {row['Total_Qty']:>10,} units{marker}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2.2  Filter Data for Selected Product
# ══════════════════════════════════════════════════════════════════════════════
TARGET_ITEM = product_sales.iloc[0]['ItemID']
TARGET_NAME = product_sales.iloc[0]['ItemName']

pdf = df[df['ItemID'] == TARGET_ITEM].sort_values('Date').reset_index(drop=True).copy()

print(f'📊 Selected: {TARGET_NAME}')
print(f'   Series length: {len(pdf)} weeks')
print(f'   Qty range: {pdf["Qty"].min():,} → {pdf["Qty"].max():,}')
print(f'   Qty mean : {pdf["Qty"].mean():,.1f} ± {pdf["Qty"].std():,.1f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2.3  Visualize the Time Series
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pdf['Week'], pdf['Qty'], 'o-', color=COLORS[0], linewidth=2, markersize=6)
ax.fill_between(pdf['Week'], pdf['Qty'], alpha=0.15, color=COLORS[0])
ax.axhline(y=pdf['Qty'].mean(), color='red', linestyle='--', alpha=0.5,
           label=f'Mean = {pdf["Qty"].mean():,.0f}')
ax.set_xlabel('Week'); ax.set_ylabel('Qty')
ax.set_title(f'Weekly Sales — {TARGET_NAME}', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

---
## 3. Feature Engineering <a id='3'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3.1  Create Lag Features
# ══════════════════════════════════════════════════════════════════════════════
pdf['lag_1'] = pdf['Qty'].shift(1)
pdf['lag_2'] = pdf['Qty'].shift(2)
pdf['lag_4'] = pdf['Qty'].shift(4)
pdf['rolling_mean_4'] = pdf['Qty'].shift(1).rolling(window=4).mean()
pdf['rolling_std_4'] = pdf['Qty'].shift(1).rolling(window=4).std()
pdf['month'] = pdf['Date'].dt.month

print('✅ Features created: lag_1, lag_2, lag_4, rolling_mean_4, rolling_std_4, month')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3.2  Visualize Features vs Target
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.flat, ['lag_1', 'lag_2', 'rolling_mean_4', 'rolling_std_4']):
    valid = pdf.dropna(subset=[col])
    ax.scatter(valid[col], valid['Qty'], alpha=0.6, s=40, color=COLORS[0])
    ax.set_xlabel(col); ax.set_ylabel('Qty')
    ax.set_title(f'{col} vs Qty', fontweight='bold')
plt.suptitle('Feature Correlation with Target', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

---
## 4. Train / Validation / Test Split (Time-Based) <a id='4'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4.1  Time-Based Split
# ══════════════════════════════════════════════════════════════════════════════
n = len(pdf)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = pdf.iloc[:train_end].copy()
val_df = pdf.iloc[train_end:val_end].copy()
test_df = pdf.iloc[val_end:].copy()

print(f'📊 Time-Based Split:')
print(f'   Train      : {len(train_df):>3} weeks  (Week {train_df["Week"].iloc[0]:>2} → {train_df["Week"].iloc[-1]:>2})  {len(train_df)/n*100:.0f}%')
print(f'   Validation : {len(val_df):>3} weeks  (Week {val_df["Week"].iloc[0]:>2} → {val_df["Week"].iloc[-1]:>2})  {len(val_df)/n*100:.0f}%')
print(f'   Test       : {len(test_df):>3} weeks  (Week {test_df["Week"].iloc[0]:>2} → {test_df["Week"].iloc[-1]:>2})  {len(test_df)/n*100:.0f}%')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4.2  Visualize the Split
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train_df['Week'], train_df['Qty'], 'o-', color=COLORS[0], lw=2, label='Train', ms=7)
ax.plot(val_df['Week'], val_df['Qty'], 's-', color=COLORS[1], lw=2, label='Validation', ms=7)
ax.plot(test_df['Week'], test_df['Qty'], 'D-', color=COLORS[3], lw=2, label='Test', ms=7)
ax.axvline(x=val_df['Week'].iloc[0]-0.5, color='gray', ls='--', alpha=0.7)
ax.axvline(x=test_df['Week'].iloc[0]-0.5, color='gray', ls='--', alpha=0.7)
ax.set_xlabel('Week'); ax.set_ylabel('Qty')
ax.set_title('Train / Validation / Test Split', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

---
## 5. Baseline Model — Naive Forecast <a id='5'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5.1  Naive Forecast on Validation Set
# ══════════════════════════════════════════════════════════════════════════════
val_df['Naive_Pred'] = val_df['Qty'].shift(1)
val_df.iloc[0, val_df.columns.get_loc('Naive_Pred')] = train_df['Qty'].iloc[-1]

print('▶ Validation — Naive Forecast:')
print(val_df[['Week', 'Qty', 'Naive_Pred']].to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5.2  Naive Forecast on Test Set
# ══════════════════════════════════════════════════════════════════════════════
test_df['Naive_Pred'] = test_df['Qty'].shift(1)
test_df.iloc[0, test_df.columns.get_loc('Naive_Pred')] = val_df['Qty'].iloc[-1]

print('▶ Test — Naive Forecast:')
print(test_df[['Week', 'Qty', 'Naive_Pred']].to_string(index=False))

---
## 6. Baseline Model — Moving Average <a id='6'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 6.1  Moving Average on Validation Set
# ══════════════════════════════════════════════════════════════════════════════
MA_WINDOW = 4

full = pd.concat([train_df[['Week', 'Qty']], val_df[['Week', 'Qty']]], ignore_index=True)
ma_v = []
for i in range(len(val_df)):
    he = len(train_df) + i
    h = full['Qty'].iloc[max(0, he - MA_WINDOW):he]
    ma_v.append(h.mean())
val_df['MA_Pred'] = ma_v

print(f'▶ Validation — Moving Average ({MA_WINDOW}w):')
print(val_df[['Week', 'Qty', 'Naive_Pred', 'MA_Pred']].to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 6.2  Moving Average on Test Set
# ══════════════════════════════════════════════════════════════════════════════
full2 = pd.concat([train_df[['Week','Qty']], val_df[['Week','Qty']],
                   test_df[['Week','Qty']]], ignore_index=True)
ma_t = []
for i in range(len(test_df)):
    he = len(train_df) + len(val_df) + i
    h = full2['Qty'].iloc[max(0, he - MA_WINDOW):he]
    ma_t.append(h.mean())
test_df['MA_Pred'] = ma_t

print(f'▶ Test — Moving Average ({MA_WINDOW}w):')
print(test_df[['Week', 'Qty', 'Naive_Pred', 'MA_Pred']].to_string(index=False))

---
## 7. Evaluation & Comparison <a id='7'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7.1  Evaluation Helper Function
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_model(y_true, y_pred, model_name):
    """Compute regression metrics for baseline models."""
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    yt = np.array(y_true)[mask]
    yp = np.array(y_pred)[mask]
    mae = mean_absolute_error(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    total = np.sum(yt)
    wmape = np.sum(np.abs(yt - yp)) / total * 100 if total > 0 else 0
    r2 = r2_score(yt, yp)
    return {'Model': model_name, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2),
            'WMAPE (%)': round(wmape, 2), 'R²': round(r2, 4)}

print('✅ evaluate_model() defined')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7.2  Evaluate on Validation Set
# ══════════════════════════════════════════════════════════════════════════════
results_val = []
results_val.append(evaluate_model(val_df['Qty'], val_df['Naive_Pred'], 'Naive Forecast'))
results_val.append(evaluate_model(val_df['Qty'], val_df['MA_Pred'], f'Moving Avg ({MA_WINDOW}w)'))

results_val_df = pd.DataFrame(results_val)
print('▶ Validation Metrics:')
print(results_val_df.to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7.3  Evaluate on Test Set
# ══════════════════════════════════════════════════════════════════════════════
results_test = []
results_test.append(evaluate_model(test_df['Qty'], test_df['Naive_Pred'], 'Naive Forecast'))
results_test.append(evaluate_model(test_df['Qty'], test_df['MA_Pred'], f'Moving Avg ({MA_WINDOW}w)'))

results_test_df = pd.DataFrame(results_test)
print('▶ Test Metrics:')
print(results_test_df.to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7.4  Visualization — Actual vs Predicted (Validation)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train_df['Week'], train_df['Qty'], 'o-', color='gray', alpha=0.4, lw=1.5, ms=5, label='Train')
ax.plot(val_df['Week'], val_df['Qty'], 'ko-', lw=2, ms=8, label='Actual (Val)')
ax.plot(val_df['Week'], val_df['Naive_Pred'], 's--', color=COLORS[0], lw=2, ms=7, label='Naive')
ax.plot(val_df['Week'], val_df['MA_Pred'], 'D--', color=COLORS[2], lw=2, ms=7, label=f'MA({MA_WINDOW}w)')
ax.axvline(x=val_df['Week'].iloc[0]-0.5, color='gray', ls='--', alpha=0.5)
ax.set_xlabel('Week'); ax.set_ylabel('Qty')
ax.set_title('Baseline vs Actual — Validation', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7.5  Visualization — Actual vs Predicted (Test)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 6))
hweeks = list(train_df['Week']) + list(val_df['Week'])
hqty = list(train_df['Qty']) + list(val_df['Qty'])
ax.plot(hweeks, hqty, 'o-', color='gray', alpha=0.4, lw=1.5, ms=4, label='History')
ax.plot(test_df['Week'], test_df['Qty'], 'ko-', lw=2, ms=8, label='Actual (Test)')
ax.plot(test_df['Week'], test_df['Naive_Pred'], 's--', color=COLORS[0], lw=2, ms=7, label='Naive')
ax.plot(test_df['Week'], test_df['MA_Pred'], 'D--', color=COLORS[2], lw=2, ms=7, label=f'MA({MA_WINDOW}w)')
ax.axvline(x=test_df['Week'].iloc[0]-0.5, color='gray', ls='--', alpha=0.5)
ax.set_xlabel('Week'); ax.set_ylabel('Qty')
ax.set_title('Baseline vs Actual — Test', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7.6  Combined Results Table
# ══════════════════════════════════════════════════════════════════════════════
print('\n' + '=' * 65)
print('  📊 BASELINE SUMMARY')
print('=' * 65)
print('\n▶ Validation:')
print(results_val_df.to_string(index=False))
print('\n▶ Test:')
print(results_test_df.to_string(index=False))

best_val = results_val_df.loc[results_val_df['MAE'].idxmin(), 'Model']
best_test = results_test_df.loc[results_test_df['MAE'].idxmin(), 'Model']
print(f'\n🏆 Best Baseline (Val) : {best_val}')
print(f'🏆 Best Baseline (Test): {best_test}')
print(f'\n💡 These baselines set the performance floor. Advanced models must beat them.')

---
## 8. Data Preparation for Global Model <a id='8'></a>

We shift from single-SKU baselines to a **global model** trained across all qualifying SKUs.

**Key decisions (data-driven):**
- **Log-transform target** — Qty has skewness=5.46 (mean=1,285, median=122). `log1p` normalizes this.
- **Filter SKUs ≥ 8 weeks** — ensures enough history for lag features.
- **Drop leakage columns** — `Total_Revenue`, `Total_Taxes` are derived from Qty (r=0.826).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 8.1  Prepare Global Dataset
# ══════════════════════════════════════════════════════════════════════════════
import json as _json
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import joblib

# Reload fresh copy
df_raw = pd.read_csv('weekly.csv', encoding='utf-8-sig')
df_raw.columns = [
    'Year', 'Week', 'ItemID', 'Qty', 'ItemName', 'BrandID', 'BrandName',
    'MasterBrandID', 'MasterBrandName', 'UOM', 'Factor',
    'Avg_Daily_Demand', 'Safety_Stock', 'ROP', 'Avg_UnitPrice',
    'Total_Promo', 'Total_CashDiscount', 'Total_ManualDiscount',
    'Total_Taxes', 'Total_Revenue'
]
df_raw['Date'] = pd.to_datetime(
    df_raw['Year'].astype(str) + '-W' + df_raw['Week'].astype(str).str.zfill(2) + '-1',
    format='%G-W%V-%u'
)

# Clip negatives (customer returns)
neg_count = (df_raw['Qty'] < 0).sum()
df_raw['Qty'] = df_raw['Qty'].clip(lower=0)

# Filter to SKUs with >= 8 weeks
week_counts = df_raw.groupby('ItemID')['Week'].count()
valid_skus = week_counts[week_counts >= 8].index
df_global = df_raw[df_raw['ItemID'].isin(valid_skus)].copy()
df_global = df_global.sort_values(['ItemID', 'Date']).reset_index(drop=True)

# Drop leakage & zero-info columns
df_global = df_global.drop(columns=[
    'Total_CashDiscount', 'Total_ManualDiscount',  # all zeros
    'Total_Revenue', 'Total_Taxes',                 # leakage (derived from Qty)
    'Avg_Daily_Demand', 'Safety_Stock', 'ROP',      # pre-calculated
    'ItemName', 'BrandName', 'MasterBrandName'      # text labels
])

# LOG TRANSFORM
df_global['Qty_log'] = np.log1p(df_global['Qty'])

print(f'✅ Global dataset prepared')
print(f'   Negatives clipped : {neg_count}')
print(f'   SKUs (≥8 weeks)   : {df_global["ItemID"].nunique()}')
print(f'   Total rows        : {len(df_global):,}')
print(f'   Skew (raw)        : {df_global["Qty"].skew():.2f}')
print(f'   Skew (log)        : {df_global["Qty_log"].skew():.2f}  ← normalized!')

---
## 9. Advanced Feature Engineering <a id='9'></a>

All lag and rolling features computed on **log scale** to match the prediction target.
A raw-scale rolling mean is kept as a bridge feature.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 9.1  Lag & Rolling Features (Per-SKU, Log Scale)
# ══════════════════════════════════════════════════════════════════════════════
df_global = df_global.sort_values(['ItemID', 'Week']).reset_index(drop=True)

# Lags on log scale
for lag in [1, 2, 4]:
    df_global[f'lag_{lag}'] = df_global.groupby('ItemID')['Qty_log'].shift(lag)

# Rolling stats on log scale
df_global['rolling_mean_4'] = df_global.groupby('ItemID')['Qty_log'].transform(
    lambda x: x.shift(1).rolling(window=4, min_periods=2).mean())
df_global['rolling_std_4'] = df_global.groupby('ItemID')['Qty_log'].transform(
    lambda x: x.shift(1).rolling(window=4, min_periods=2).std()).fillna(0)

# Raw-scale rolling mean (bridge feature)
df_global['rolling_mean_4_raw'] = df_global.groupby('ItemID')['Qty'].transform(
    lambda x: x.shift(1).rolling(window=4, min_periods=2).mean())

print('✅ Lag features: lag_1, lag_2, lag_4 (log scale)')
print('✅ Rolling: rolling_mean_4, rolling_std_4 (log), rolling_mean_4_raw')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 9.2  Price, Promotional & Temporal Features
# ══════════════════════════════════════════════════════════════════════════════
df_global['price_change_pct'] = df_global.groupby('ItemID')['Avg_UnitPrice'] \
    .pct_change().fillna(0).clip(-1, 1)
df_global['log_price'] = np.log1p(df_global['Avg_UnitPrice'])
df_global['has_promo'] = (df_global['Total_Promo'] > 0).astype(int)
df_global['Month'] = df_global['Date'].dt.month
df_global['week_sin'] = np.sin(2 * np.pi * df_global['Week'] / 52)
df_global['week_cos'] = np.cos(2 * np.pi * df_global['Week'] / 52)
df_global['UOM_encoded'] = (df_global['UOM'] == 'PL').astype(int)

print('✅ All features engineered')
print(f'   Total columns: {df_global.shape[1]}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 9.3  Feature Correlation Heatmap
# ══════════════════════════════════════════════════════════════════════════════
feat_cols = ['Qty_log', 'lag_1', 'lag_2', 'lag_4', 'rolling_mean_4', 'rolling_std_4',
             'rolling_mean_4_raw', 'Avg_UnitPrice', 'log_price', 'price_change_pct',
             'has_promo', 'Total_Promo', 'Factor', 'Week', 'Month', 'UOM_encoded']
corr_df = df_global[feat_cols].dropna()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_df.corr(), dtype=bool))
sns.heatmap(corr_df.corr(), mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()

print('\nTop correlations with Qty_log:')
print(corr_df.corr()['Qty_log'].drop('Qty_log').sort_values(ascending=False).to_string())

---
## 10. Train / Val / Test Split + Target Encoding <a id='10'></a>

Chronological split. Target encoding computed from **training data only** to prevent leakage.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 10.1  Chronological Split
# ══════════════════════════════════════════════════════════════════════════════
df_model = df_global.dropna(subset=['lag_1', 'lag_2', 'lag_4', 'rolling_mean_4']).copy()

train_g = df_model[df_model['Week'] <= 18].copy()
val_g   = df_model[(df_model['Week'] >= 19) & (df_model['Week'] <= 22)].copy()
test_g  = df_model[df_model['Week'] >= 23].copy()

total_g = len(train_g) + len(val_g) + len(test_g)
print(f'📊 Global Split:')
print(f'   Train : {len(train_g):>5} rows  (Weeks  1–18)  {len(train_g)/total_g*100:.0f}%')
print(f'   Val   : {len(val_g):>5} rows  (Weeks 19–22)  {len(val_g)/total_g*100:.0f}%')
print(f'   Test  : {len(test_g):>5} rows  (Weeks 23–26)  {len(test_g)/total_g*100:.0f}%')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 10.2  Target Encoding (log-scale, train-only)
# ══════════════════════════════════════════════════════════════════════════════
item_target_enc = train_g.groupby('ItemID')['Qty_log'].mean().to_dict()
global_mean_qty = train_g['Qty_log'].mean()
brand_target_enc = train_g.groupby('BrandID')['Qty_log'].mean().to_dict()
brand_avg_week = train_g.groupby(['BrandID', 'Week'])['Qty_log'].mean() \
    .groupby('BrandID').mean().to_dict()

for s in [train_g, val_g, test_g]:
    s['item_encoded']       = s['ItemID'].map(item_target_enc).fillna(global_mean_qty)
    s['brand_encoded']      = s['BrandID'].map(brand_target_enc).fillna(global_mean_qty)
    s['brand_avg_week_qty'] = s['BrandID'].map(brand_avg_week).fillna(global_mean_qty)

print('✅ Target encoding applied (log-scale, train-only)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 10.3  Define Feature Columns & Prepare Matrices
# ══════════════════════════════════════════════════════════════════════════════
FEATURE_COLS = [
    'lag_1', 'lag_2', 'lag_4', 'rolling_mean_4', 'rolling_std_4',
    'rolling_mean_4_raw',
    'Avg_UnitPrice', 'log_price', 'price_change_pct',
    'Total_Promo', 'has_promo',
    'Week', 'Month', 'week_sin', 'week_cos',
    'Factor', 'UOM_encoded',
    'item_encoded', 'brand_encoded', 'brand_avg_week_qty'
]
TARGET = 'Qty_log'

# Verify no leakage
leakage = {'Total_Revenue', 'Total_Taxes', 'Qty'}
assert leakage.isdisjoint(set(FEATURE_COLS)), '🚨 LEAKAGE!'

X_train, y_train = train_g[FEATURE_COLS], train_g[TARGET]
X_val,   y_val   = val_g[FEATURE_COLS],   val_g[TARGET]
X_test,  y_test  = test_g[FEATURE_COLS],  test_g[TARGET]

y_val_real  = val_g['Qty'].values
y_test_real = test_g['Qty'].values

print(f'✅ {len(FEATURE_COLS)} features, target: {TARGET}')
print(f'   X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')
print(f'🔒 Leakage check passed')

---
## 11. Model Training <a id='11'></a>

Models predict `log1p(Qty)`. Predictions are converted back via `expm1()`.

$$\text{WMAPE} = \frac{\sum |\text{actual} - \text{predicted}|}{\sum \text{actual}} \times 100$$

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 11.1  Helpers
# ══════════════════════════════════════════════════════════════════════════════

def to_real(log_preds):
    """Convert log predictions back to real units."""
    return np.clip(np.expm1(np.clip(log_preds, 0, None)), 0, None)

def evaluate_advanced(y_real, pred_real, name):
    """Evaluate with WMAPE on real-scale values."""
    mae  = mean_absolute_error(y_real, pred_real)
    rmse = np.sqrt(mean_squared_error(y_real, pred_real))
    r2   = r2_score(y_real, pred_real)
    total = np.sum(y_real)
    wmape = np.sum(np.abs(y_real - pred_real)) / total * 100 if total > 0 else 0
    print(f'  {name:<25} MAE={mae:>10,.1f}   RMSE={rmse:>10,.1f}   WMAPE={wmape:>6.1f}%   R²={r2:.4f}')
    return {'Model': name, 'MAE': round(mae, 1), 'RMSE': round(rmse, 1),
            'WMAPE (%)': round(wmape, 2), 'R²': round(r2, 4), '_wmape': wmape}

print('✅ Helpers defined')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 11.2  XGBoost Regressor
# ══════════════════════════════════════════════════════════════════════════════
print('=' * 65)
print('  🌲 XGBoost Regressor')
print('=' * 65)

xgb_model = XGBRegressor(
    n_estimators=500, max_depth=5, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.5, reg_lambda=2.0, min_child_weight=3,
    random_state=42, n_jobs=-1, verbosity=0
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

xgb_val_real  = to_real(xgb_model.predict(X_val))
xgb_test_real = to_real(xgb_model.predict(X_test))

print('\n▶ Validation:')
xgb_m = evaluate_advanced(y_val_real, xgb_val_real, 'XGBoost')

# Feature Importance
fi = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(10, 7))
fi.plot.barh(ax=ax, color=COLORS[0])
ax.set_title('XGBoost — Feature Importance', fontweight='bold')
ax.set_xlabel('Importance'); plt.tight_layout(); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 11.3  Random Forest Regressor
# ══════════════════════════════════════════════════════════════════════════════
print('=' * 65)
print('  🌳 Random Forest Regressor')
print('=' * 65)

rf_model = RandomForestRegressor(
    n_estimators=300, max_depth=12, min_samples_leaf=3,
    random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_val_real  = to_real(rf_model.predict(X_val))
rf_test_real = to_real(rf_model.predict(X_test))

print('\n▶ Validation:')
rf_m = evaluate_advanced(y_val_real, rf_val_real, 'Random Forest')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 11.4  Ridge Regression
# ══════════════════════════════════════════════════════════════════════════════
print('=' * 65)
print('  📐 Ridge Regression')
print('=' * 65)

ridge_scaler = StandardScaler()
X_train_s = ridge_scaler.fit_transform(X_train)
X_val_s   = ridge_scaler.transform(X_val)
X_test_s  = ridge_scaler.transform(X_test)

ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_s, y_train)

ridge_val_real  = to_real(ridge_model.predict(X_val_s))
ridge_test_real = to_real(ridge_model.predict(X_test_s))

print('\n▶ Validation:')
ridge_m = evaluate_advanced(y_val_real, ridge_val_real, 'Ridge Regression')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 11.5  Weighted Ensemble (XGBoost + Random Forest)
# ══════════════════════════════════════════════════════════════════════════════
print('=' * 65)
print('  🏆 Weighted Ensemble (XGB + RF)')
print('=' * 65)

inv_sum = (1.0 / xgb_m['_wmape']) + (1.0 / rf_m['_wmape'])
w_xgb = (1.0 / xgb_m['_wmape']) / inv_sum
w_rf  = (1.0 / rf_m['_wmape']) / inv_sum
ensemble_weights = {'XGBoost': w_xgb, 'Random Forest': w_rf}

ens_val_real  = w_xgb * xgb_val_real  + w_rf * rf_val_real
ens_test_real = w_xgb * xgb_test_real + w_rf * rf_test_real

print(f'   XGBoost weight : {w_xgb:.3f}')
print(f'   RF weight      : {w_rf:.3f}')
print('\n▶ Validation:')
ens_m = evaluate_advanced(y_val_real, ens_val_real, '⭐ XGB+RF Ensemble')

---
## 12. Final Evaluation on Test Set <a id='12'></a>

Weeks 23–26 — **never seen** during training or weight tuning.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 12.1  Test Metrics — All Models
# ══════════════════════════════════════════════════════════════════════════════
print('=' * 65)
print('  📊 FINAL TEST SET EVALUATION')
print('=' * 65 + '\n')

test_results = []
test_results.append(evaluate_advanced(y_test_real, xgb_test_real,   'XGBoost'))
test_results.append(evaluate_advanced(y_test_real, rf_test_real,    'Random Forest'))
test_results.append(evaluate_advanced(y_test_real, ridge_test_real, 'Ridge Regression'))
test_results.append(evaluate_advanced(y_test_real, ens_test_real,   '⭐ XGB+RF Ensemble'))

results_adv = pd.DataFrame(test_results).drop(columns=['_wmape'])
print('\n' + '─' * 70)
print(results_adv.to_string(index=False))
print('─' * 70)

best_name = results_adv.loc[results_adv['WMAPE (%)'].idxmin(), 'Model']
best_wmape = results_adv['WMAPE (%)'].min()
best_r2 = results_adv.loc[results_adv['WMAPE (%)'].idxmin(), 'R²']
print(f'\n🏆 Best: {best_name} — WMAPE={best_wmape:.1f}%, R²={best_r2:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 12.2  Actual vs Predicted Scatter Plots
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, preds) in zip(axes, [('XGBoost', xgb_test_real),
                                     ('Random Forest', rf_test_real),
                                     ('XGB+RF Ensemble', ens_test_real)]):
    ax.scatter(y_test_real, preds, alpha=0.4, s=20, color=COLORS[0])
    mx = max(y_test_real.max(), preds.max())
    ax.plot([0, mx], [0, mx], 'r--', alpha=0.7, label='Perfect')
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.set_title(name, fontweight='bold'); ax.legend()
plt.suptitle('Actual vs Predicted — Test Set', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 12.3  Top 5 SKUs — Time Series Comparison
# ══════════════════════════════════════════════════════════════════════════════
top5 = df_global.groupby('ItemID')['Qty'].sum().nlargest(5).index.tolist()
fig, axes = plt.subplots(5, 1, figsize=(14, 20))

test_plot = test_g.copy()
test_plot['Ens_Pred'] = ens_test_real

for sku, ax in zip(top5, axes):
    hist = pd.concat([train_g[train_g['ItemID']==sku],
                      val_g[val_g['ItemID']==sku]])[['Week','Qty']].sort_values('Week')
    tst = test_plot[test_plot['ItemID']==sku][['Week','Qty','Ens_Pred']].sort_values('Week')
    ax.plot(hist['Week'], hist['Qty'], 'o-', color='gray', alpha=0.5, lw=1.5, ms=4, label='History')
    ax.plot(tst['Week'], tst['Qty'], 'ko-', lw=2, ms=7, label='Actual')
    ax.plot(tst['Week'], tst['Ens_Pred'], 'D--', color=COLORS[2], lw=2, ms=7, label='Ensemble')
    ax.axvline(x=22.5, color='red', ls='--', alpha=0.5)
    ax.set_ylabel('Qty'); ax.set_title(f'SKU: {sku}', fontweight='bold')
    ax.legend(loc='upper left', fontsize=8)
axes[-1].set_xlabel('Week')
plt.suptitle('Top 5 SKUs — Actual vs Ensemble', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 12.4  Residuals & Per-Brand WMAPE
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

residuals = y_test_real - ens_test_real
axes[0].hist(residuals, bins=40, color=COLORS[0], edgecolor='white', alpha=0.8)
axes[0].axvline(x=0, color='red', ls='--', alpha=0.7)
axes[0].set_xlabel('Residual'); axes[0].set_ylabel('Frequency')
axes[0].set_title('Residual Distribution', fontweight='bold')

teval = test_g.copy(); teval['Pred'] = ens_test_real
bmap = df_raw[['ItemID','BrandName']].drop_duplicates()
teval = teval.merge(bmap, on='ItemID', how='left')
bmape = []
for brand, g in teval.groupby('BrandName'):
    s = g['Qty'].sum()
    if s > 0: bmape.append({'Brand': brand, 'WMAPE': np.sum(np.abs(g['Qty']-g['Pred']))/s*100})
bm = pd.DataFrame(bmape).sort_values('WMAPE')
axes[1].barh(bm['Brand'], bm['WMAPE'], color=COLORS[1])
axes[1].set_xlabel('WMAPE (%)'); axes[1].set_title('Per-Brand WMAPE', fontweight='bold')
plt.tight_layout(); plt.show()

---
## 13. Model Serialization & Export <a id='13'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 13.1  Save All Artifacts
# ══════════════════════════════════════════════════════════════════════════════
MODEL_DIR = Path('..') / 'src' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(xgb_model,    MODEL_DIR / 'xgb_model.joblib')
joblib.dump(rf_model,     MODEL_DIR / 'rf_model.joblib')
joblib.dump(ridge_model,  MODEL_DIR / 'ridge_model.joblib')
joblib.dump(ridge_scaler, MODEL_DIR / 'ridge_scaler.joblib')

with open(MODEL_DIR / 'ensemble_weights.json', 'w') as f: _json.dump(ensemble_weights, f, indent=2)
with open(MODEL_DIR / 'feature_cols.json', 'w') as f: _json.dump(FEATURE_COLS, f, indent=2)
with open(MODEL_DIR / 'item_encoder.json', 'w') as f: _json.dump(item_target_enc, f, indent=2)
with open(MODEL_DIR / 'brand_encoder.json', 'w') as f: _json.dump(brand_target_enc, f, indent=2)
with open(MODEL_DIR / 'brand_avg_week.json', 'w') as f: _json.dump(brand_avg_week, f, indent=2)
with open(MODEL_DIR / 'global_mean_qty.json', 'w') as f:
    _json.dump({'global_mean_qty': global_mean_qty}, f, indent=2)

print('✅ Artifacts saved:')
for p in sorted(MODEL_DIR.glob('*')):
    if p.name != '.gitkeep':
        print(f'   {p.name}  ({p.stat().st_size / 1024:.1f} KB)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 13.2  Verification
# ══════════════════════════════════════════════════════════════════════════════
xgb_ld = joblib.load(MODEL_DIR / 'xgb_model.joblib')
sample_pred = to_real(xgb_ld.predict(X_test.head(5)))
print('🔍 Verification:')
print(f'   Predicted : {np.round(sample_pred, 1)}')
print(f'   Actual    : {test_g["Qty"].head(5).values}')
print('✅ Models verified!')

---
## 14. Export Forecasts for Inventory Optimization <a id='14'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 14.1  Generate Forecasts & Inventory Metrics
# ══════════════════════════════════════════════════════════════════════════════
LEAD_TIME = 7; Z = 1.65  # 95% service level

all_data = pd.concat([train_g, val_g, test_g], ignore_index=True)
X_all = all_data[FEATURE_COLS]
all_data['pred_ens'] = w_xgb * to_real(xgb_model.predict(X_all)) + \
                       w_rf  * to_real(rf_model.predict(X_all))

sku_fc = all_data.groupby('ItemID').agg(
    forecast_mean=('pred_ens', 'mean'), forecast_std=('pred_ens', 'std'),
    actual_mean=('Qty', 'mean'), weeks=('Week', 'count')
).reset_index()
sku_fc['forecast_std'] = sku_fc['forecast_std'].fillna(0)
sku_fc['avg_daily_demand_ml'] = sku_fc['forecast_mean'] / 7
sku_fc['safety_stock_ml'] = Z * sku_fc['forecast_std'] * np.sqrt(LEAD_TIME / 7)
sku_fc['rop_ml'] = sku_fc['avg_daily_demand_ml'] * LEAD_TIME + sku_fc['safety_stock_ml']

nm = df_raw[['ItemID', 'ItemName']].drop_duplicates()
sku_fc = sku_fc.merge(nm, on='ItemID', how='left')

PROC_DIR = Path('..') / 'data' / 'processed'
PROC_DIR.mkdir(parents=True, exist_ok=True)
sku_fc.to_csv(PROC_DIR / 'ml_forecasts.csv', index=False, encoding='utf-8-sig')

print(f'✅ {len(sku_fc)} SKUs exported to data/processed/ml_forecasts.csv')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 14.2  ML vs Original Inventory Comparison
# ══════════════════════════════════════════════════════════════════════════════
orig = df_raw.groupby('ItemID').agg(
    orig_ss=('Safety_Stock', 'mean'), orig_rop=('ROP', 'mean')).reset_index()
comp = sku_fc.merge(orig, on='ItemID', how='left')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (oc, mc, t) in zip(axes, [('orig_ss','safety_stock_ml','Safety Stock'),
                                    ('orig_rop','rop_ml','Reorder Point')]):
    ax.scatter(comp[oc], comp[mc], alpha=0.5, s=30, color=COLORS[0])
    mx = max(comp[oc].max(), comp[mc].max())
    ax.plot([0, mx], [0, mx], 'r--', alpha=0.7)
    ax.set_xlabel(f'Original {t}'); ax.set_ylabel(f'ML {t}')
    ax.set_title(f'{t}: Original vs ML', fontweight='bold')
plt.tight_layout(); plt.show()

print('\n' + '=' * 65)
print('  ✅ MILESTONE 2 COMPLETE')
print('=' * 65)
print(f'  Best model : {best_name} — WMAPE={best_wmape:.1f}%, R²={best_r2:.4f}')
print(f'  Artifacts  → src/models/')
print(f'  Forecasts  → data/processed/ml_forecasts.csv')
print(f'  🚀 Ready for FastAPI & mobile app deployment!')